# Week 4 — Creative Content & Gen AI Hub
**Tools:** Gemini 1.5 Flash API, NLTK, Sentence Transformers
**Datasets:**
- sloganlist.csv — columns: `Company`, `Slogan` (1,162 rows)
- startups.csv — columns: `name`, `city`, `tagline`, `description` (42,038 rows)

In [ ]:
!pip install google-generativeai nltk sentence-transformers pandas -q
import pandas as pd,numpy as np,json,re,nltk
from collections import Counter
import google.generativeai as genai
from google.colab import userdata,drive
from sklearn.metrics.pairwise import cosine_similarity
nltk.download('punkt',quiet=True); nltk.download('stopwords',quiet=True)
from nltk.corpus import stopwords as nltk_stop; from nltk.tokenize import word_tokenize
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
gm=genai.GenerativeModel('gemini-1.5-flash')
print('Gemini 1.5 Flash ready ✓')

In [ ]:
drive.mount('/content/drive')
BASE='/content/drive/MyDrive/BrandSphere/'
# Slogans: Company | Slogan
sl=pd.read_csv(BASE+'sloganlist.csv'); sl.columns=['Company','Slogan']
sl['Slogan']=sl['Slogan'].astype(str).str.strip(); sl=sl[sl['Slogan'].str.len()>3].dropna()
print(f'Slogans: {len(sl)} | Columns: {list(sl.columns)}'); display(sl.head(3))
# Startups: name | city | tagline | description
st=pd.read_csv(BASE+'startups.csv').dropna(subset=['tagline'])
print(f'Startups: {len(st)} | Columns: {list(st.columns)}'); display(st[['name','city','tagline']].head(3))

In [ ]:
# NLTK preprocessing on slogan column
stop_en=set(nltk_stop.words('english'))
sl['cleaned']=sl['Slogan'].str.lower().str.replace(r'[^\w\s]',' ',regex=True).str.strip()
sl['tokens']=sl['cleaned'].apply(lambda x:[w for w in word_tokenize(x) if w not in stop_en and len(w)>2])
sl['word_count']=sl['tokens'].apply(len)
all_tok=[w for toks in sl['tokens'] for w in toks]
vocab=Counter(all_tok)
print(f'Avg words/slogan: {sl["word_count"].mean():.1f} | Vocab: {len(vocab)}')
print('Top 15:',vocab.most_common(15))

In [ ]:
def generate_taglines(company,industry,audience,tone,product,n=5,refs=None):
    ref_block=''
    if refs: ref_block='\n\nReal slogans for STYLE inspiration only:\n'+"\n".join(f'  - {s}' for s in refs[:12])
    prompt=f"""You are a world-class brand strategist.\nBrand: {company} ({industry})\nProduct: {product}\nAudience: {audience} | Tone: {tone}{ref_block}\n\nGenerate {n} ORIGINAL taglines. Rules: 3–8 words | specific to {industry} | match {tone}\nAvoid: excellence, world-class, leading, innovative, revolutionize\nReturn ONLY valid JSON array: [\"t1\",\"t2\"]"""
    raw=gm.generate_content(prompt).text.strip(); m=re.search(r'\[.*?\]',raw,re.DOTALL)
    return json.loads(m.group()) if m else [raw]

refs=sl['Slogan'].sample(15,random_state=42).tolist()
taglines=generate_taglines('NovaTech','Technology','Tech professionals 25-40','Tech-Forward & Minimalist','AI enterprise automation software',6,refs)
print('Generated taglines:'); [print(f'  {i}. {t}') for i,t in enumerate(taglines,1)]

In [ ]:
def gen_narrative(company,industry,product,tone,audience):
    p=f"""3-paragraph brand narrative for {company} ({industry}). Product: {product}. Tone: {tone}. Audience: {audience}.\nPara1: origin & mission | Para2: differentiation | Para3: brand promise. Under 170 words. Plain text."""
    return gm.generate_content(p).text.strip()
def gen_positioning(company,industry,product,audience):
    p=f"""ONE positioning sentence: \"For [audience] who [need], {company} is the [category] that [benefit] because [reason].\"\nContext: {product}. Under 40 words. Plain text."""
    return gm.generate_content(p).text.strip()
narrative=gen_narrative('NovaTech','Technology','AI enterprise automation','Tech-Forward','Professionals 25-40')
positioning=gen_positioning('NovaTech','Technology','AI enterprise automation','Tech-savvy leaders')
print('NARRATIVE:\n',narrative,'\n\nPOSITIONING:\n',positioning)

In [ ]:
def translate(tagline,langs):
    p=f"""Translate \"${tagline}\" into: {', '.join(langs)}.\nBrand: NovaTech | Tone: Tech-Forward.\nSound natural, preserve emotional tone, adapt culturally.\nReturn ONLY JSON: {{{', '.join(f\"{l}\": \"...\"\" for l in langs)}}}"""
    raw=gm.generate_content(p).text.strip(); m=re.search(r'\{.*?\}',raw,re.DOTALL)
    return json.loads(m.group()) if m else {}
translations=translate(taglines[0],['Hindi','Spanish','French','Mandarin','Arabic'])
print(f'Original: {taglines[0]}'); [print(f'  {l}: {t}') for l,t in translations.items()]

In [ ]:
# Semantic similarity search — find similar slogans from dataset
from sentence_transformers import SentenceTransformer
encoder=SentenceTransformer('all-MiniLM-L6-v2')
sl_embeds=encoder.encode(sl['Slogan'].tolist(),batch_size=64,show_progress_bar=True)
tl_embeds=encoder.encode(taglines)
print('\nTop 3 similar dataset slogans per tagline:')
for tl,te in zip(taglines,tl_embeds):
    sims=cosine_similarity([te],sl_embeds)[0]; top3=sims.argsort()[-3:][::-1]
    print(f'\n  "{tl}"'); [print(f'    [{sims[i]:.3f}] {sl["Slogan"].iloc[i]}') for i in top3]

In [ ]:
output={'company':'NovaTech','taglines':taglines,'narrative':narrative,'positioning':positioning,'translations':translations}
with open('week4_output.json','w',encoding='utf-8') as f: json.dump(output,f,ensure_ascii=False,indent=2)
print('Saved week4_output.json ✓')

## ✅ Week 4 Deliverables
- [ ] sloganlist.csv loaded (Company, Slogan columns) — NLTK tokenised
- [ ] startups.csv loaded (name, city, tagline, description columns)
- [ ] Gemini 1.5 Flash: 6 taglines, narrative, positioning statement
- [ ] Multilingual: Hindi, Spanish, French, Mandarin, Arabic
- [ ] Sentence Transformers similarity search on 1,162 slogans
- [ ] All outputs saved: week4_output.json